# U4-01 · Extracción y diagnóstico de calidad

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 4, apartados 4.2 y 4.3*

Ejecuta cada celda con **Mayús + Intro**. Todos los datos son ficticios y se crean dentro del propio notebook.

## 1. Leer CSV y JSON

In [ ]:
import io
import pandas as pd

csv_txt = """id,linea,fecha,piezas
1,L1,2026-03-02,410
2,L2,2026-03-02,520
3,L1,2026-03-03,385"""
df = pd.read_csv(io.StringIO(csv_txt), parse_dates=["fecha"])
print(df.dtypes)
df

In [ ]:
json_txt = '[{"id": 1, "linea": "L1", "fecha": "2026-03-02", "piezas": 410}, {"id": 2, "linea": "L2", "fecha": "2026-03-02", "piezas": 520}]'
pd.read_json(io.StringIO(json_txt))

## 2. El CSV «a la española» (ejemplo 4.1)

In [ ]:
es = "linea;consumo_kwh\nL1;12,5\nL2;9,8"
mal = pd.read_csv(io.StringIO(es))
print("Opciones por defecto:", mal.shape, list(mal.columns))
bien = pd.read_csv(io.StringIO(es), sep=";", decimal=",")
print("Con sep y decimal:  ", bien.shape, bien.dtypes.to_dict())

También puedes escribir el fichero y leerlo como harías con uno real:

In [ ]:
with open("consumo.csv", "w", encoding="utf-8") as f:
    f.write(es)
pd.read_csv("consumo.csv", sep=";", decimal=",")

## 3. Consultar una base de datos SQL

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
df.to_sql("produccion", con, index=False)
pd.read_sql("SELECT linea, SUM(piezas) AS total FROM produccion GROUP BY linea", con)

**Ejercicio.** Escribe una consulta SQL que devuelva solo los registros con más de 400 piezas, ordenados de mayor a menor.

## 4. Informe de calidad (ejemplo 4.2)

In [ ]:
import pandas as pd
import numpy as np
datos = pd.DataFrame({
    "fecha":    ["2026-03-02", "2026-03-02", "2026-03-03", "2026-03-03",
                 "2026-03-04", "2026-03-04", "2026-03-04", "2026-03-05"],
    "linea":    ["L1", "L2", "L1", "l2 ", "L1", "L2", "L2", "L1"],
    "piezas":   [410, 520, 385, 498, None, 505, 505, 402],
    "temp_c":   [21.5, 22.0, 250.0, 21.8, 22.4, 23.1, 23.1, None],
    "defectos": [6, 4, 9, 7, 5, None, None, 8],
})
datos

In [ ]:
print("Ausentes por columna:", datos.isna().sum().to_dict())
print("Completitud (%):", (100 * (1 - datos.isna().mean())).round(1).to_dict())
print("Filas duplicadas:", datos.duplicated().sum())
print("Líneas:", datos["linea"].value_counts().to_dict())

In [ ]:
datos.describe().loc[["min", "max"]]

In [ ]:
def informe_calidad(df, rangos):
    """Devuelve una tabla con completitud y valores fuera de rango por columna."""
    filas = []
    for col in df.columns:
        fuera = None
        if col in rangos:
            lo, hi = rangos[col]
            fuera = int(((df[col] < lo) | (df[col] > hi)).sum())
        filas.append({"columna": col, "ausentes": int(df[col].isna().sum()),
                      "completitud_%": round(100 * df[col].notna().mean(), 1),
                      "fuera_de_rango": fuera})
    return pd.DataFrame(filas)

informe_calidad(datos, {"piezas": (0, 1000), "temp_c": (-10, 60), "defectos": (0, 100)})

**Ejercicio.** Añade al informe una columna con el número de valores distintos de cada variable. ¿Qué te dice sobre `linea`?

> Wang, R. Y. y Strong, D. M. (1996). Beyond Accuracy: What Data Quality Means to Data Consumers. *JMIS*, 12(4), 5–33. https://doi.org/10.1080/07421222.1996.11518099